**Filter matches to consider games after 2000**

In [0]:
from pyspark.sql import functions as F

RAW = "bq_raw_statsbomb_sa_catalog.raw_statsbomb"

m = spark.table(f"{RAW}.matches")
for c in ("season", "competition", "competition_stage", "home_team", "away_team",
          "stadium", "referee", "home_managers", "away_managers"):
    if c in m.columns:
        m = m.drop(c)

m.createOrReplaceTempView("matches_no_structs")

spark.sql("""
CREATE OR REPLACE TEMP VIEW matches_after_2000 AS
SELECT
  match_id,
  match_date,
  competition_name,
  season_name,
  gender,
  is_youth,
  is_international,
  home_team_id,
  away_team_id,
  home_score,
  away_score
FROM matches_no_structs
WHERE season_name IS NOT NULL
  AND CAST(regexp_extract(season_name, '^\\\\d{4}', 0) AS INT) >= 2000
""")

matches_after_2000 = spark.table("matches_after_2000")

**Filter based on the competitions/season that have at least 10 games**

In [0]:
from pyspark.sql import functions as F

comp_season_ge_10 = (
    matches_after_2000.groupBy("competition_name", "season_name")
    .agg(F.countDistinct("match_id").alias("n_matches"))
    .filter(F.col("n_matches") >= 10)
    .select("competition_name", "season_name")
)

filtered_matches = matches_after_2000.join(
    comp_season_ge_10,
    on=["competition_name", "season_name"],
    how="inner",
)

filtered_matches.createOrReplaceTempView("filtered_matches")

**Filter players who have played 270 minutes or more**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

RAW = "bq_raw_statsbomb_sa_catalog.raw_statsbomb"

# Same clock parsing idea as player_club_vs_national.ipynb (cell 7)
def parse_time_to_minutes(t):
    if t is None:
        return None
    s = str(t).strip()
    if s == "" or s.lower() == "nan":
        return None
    parts = s.split(":")
    try:
        if len(parts) == 2:
            return int(parts[0]) + int(parts[1]) / 60.0
        if len(parts) == 3:
            return int(parts[0]) * 60 + int(parts[1]) + int(parts[2]) / 60.0
    except (ValueError, TypeError):
        return None
    return None

parse_udf = F.udf(parse_time_to_minutes, DoubleType())

# Join lineups to matches with disambiguated column names (BQ lineups often duplicate match fields).
fm = filtered_matches.select(
    F.col("match_id"),
    F.col("competition_name").alias("match_competition_name"),
    F.col("season_name").alias("match_season_name"),
    F.col("is_international").alias("match_is_international"),
    F.col("gender").alias("match_gender"),
    F.col("is_youth").alias("match_is_youth"),
).distinct()
l = spark.table(f"{RAW}.lineups")

minutes_df = (
    l.join(fm, on="match_id", how="inner")
    .filter(F.col("player_name").isNotNull())
    .filter((F.col("match_gender") == F.lit("male")) & (F.col("match_is_youth") == F.lit(False)))
    .withColumn(
        "context",
        F.when(F.col("match_is_international"), F.lit("National")).otherwise(F.lit("Club")),
    )
    .withColumn("from_min", F.coalesce(parse_udf(F.col("from_time")), F.lit(0.0)))
    .withColumn(
        "to_min",
        F.least(F.lit(120.0), F.coalesce(parse_udf(F.col("to_time")), F.lit(90.0))),
    )
    .withColumn(
        "minutes_played",
        F.greatest(F.lit(0.0), F.col("to_min") - F.col("from_min")),
    )
)

# Some lineups tables also carry competition/season; keep the match-table versions only.
if "competition_name" in minutes_df.columns and "match_competition_name" in minutes_df.columns:
    minutes_df = minutes_df.drop("competition_name")
if "season_name" in minutes_df.columns and "match_season_name" in minutes_df.columns:
    minutes_df = minutes_df.drop("season_name")

player_comp_season_minutes = (
    minutes_df.filter(F.col("minutes_played").isNotNull())
    .withColumnRenamed("match_competition_name", "competition_name")
    .withColumnRenamed("match_season_name", "season_name")
    .groupBy("player_name", "competition_name", "season_name")
    .agg(F.sum("minutes_played").alias("minutes"))
)

eligible_player_competition_seasons = player_comp_season_minutes.filter(F.col("minutes") >= 270)
eligible_player_competition_seasons.createOrReplaceTempView("eligible_player_competition_seasons")

Build player aggregates for RQ3

In [0]:
# player_club_vs_national.ipynb cell 11 + 13: aggregate events at (player, team, match_id, context),
# then sum to (player, context). Same filters as DuckDB (male, not youth).

RAW = "bq_raw_statsbomb_sa_catalog.raw_statsbomb"

player_team_match_event_totals = spark.sql(f"""
WITH enriched AS (
    SELECT
        TRIM(CAST(e.player AS STRING)) AS player,
        TRIM(CAST(e.team AS STRING)) AS team,
        e.match_id,
        CASE WHEN m.is_international THEN 'National' ELSE 'Club' END AS context,
        LOWER(TRIM(COALESCE(
            NULLIF(GET_JSON_OBJECT(CAST(e.`type` AS STRING), '$.name'), ''),
            CAST(e.`type` AS STRING)
        ))) AS et,
        e.shot_statsbomb_xg,
        e.pass_outcome,
        e.location_x
    FROM {RAW}.events e
    INNER JOIN filtered_matches m ON e.match_id = m.match_id
    WHERE e.player IS NOT NULL
      AND m.gender = 'male'
      AND m.is_youth = false
),
per_team_match AS (
    SELECT
        player,
        team,
        match_id,
        context,
        SUM(CASE WHEN et = 'shot' THEN 1 ELSE 0 END) AS shots,
        SUM(CASE WHEN et = 'shot' THEN shot_statsbomb_xg END) AS xg,
        SUM(CASE WHEN et = 'pass' THEN 1 ELSE 0 END) AS passes,
        SUM(CASE WHEN et = 'pass' AND pass_outcome IS NULL THEN 1 ELSE 0 END) AS passes_completed,
        SUM(CASE WHEN et = 'pass' AND location_x > 80 THEN 1 ELSE 0 END) AS passes_att_third,
        SUM(CASE WHEN et = 'pressure' THEN 1 ELSE 0 END) AS pressures,
        SUM(CASE WHEN et = 'carry' THEN 1 ELSE 0 END) AS carries,
        SUM(CASE WHEN et = 'carry' AND location_x > 80 THEN 1 ELSE 0 END) AS carries_att_third,
        SUM(CASE WHEN et = 'duel' THEN 1 ELSE 0 END) AS duels,
        SUM(CASE WHEN et = 'dribble' THEN 1 ELSE 0 END) AS dribbles,
        SUM(CASE WHEN et = 'interception' THEN 1 ELSE 0 END) AS interceptions,
        SUM(CASE WHEN et = 'clearance' THEN 1 ELSE 0 END) AS clearances,
        SUM(CASE WHEN et = 'ball receipt*' THEN 1 ELSE 0 END) AS ball_receipts
    FROM enriched
    GROUP BY player, team, match_id, context
)
SELECT
    player,
    context,
    SUM(shots) AS shots,
    SUM(xg) AS xg,
    SUM(passes) AS passes,
    SUM(passes_completed) AS passes_completed,
    SUM(passes_att_third) AS passes_att_third,
    SUM(pressures) AS pressures,
    SUM(carries) AS carries,
    SUM(carries_att_third) AS carries_att_third,
    SUM(duels) AS duels,
    SUM(dribbles) AS dribbles,
    SUM(interceptions) AS interceptions,
    SUM(clearances) AS clearances,
    SUM(ball_receipts) AS ball_receipts
FROM per_team_match
GROUP BY player, context
""")

player_team_match_event_totals.createOrReplaceTempView("player_team_match_event_totals")

Validation on number of matches

In [0]:
%sql
SELECT
  CASE WHEN is_international THEN 'National' ELSE 'Club' END AS context,
  COUNT(DISTINCT match_id) AS matches
FROM filtered_matches
GROUP BY is_international

Validation on player's metrics

In [0]:
ctx_summary = spark.sql("""
SELECT
  context,
  COUNT(DISTINCT player) AS players,
  SUM(shots) AS shots,
  SUM(xg) AS xg,
  SUM(passes) AS passes,
  SUM(pressures) AS pressures
FROM player_team_match_event_totals
GROUP BY context
""")
display(ctx_summary)

In [0]:
import matplotlib.pyplot as plt

CLR_CLUB = "#4C72B0"
CLR_NATIONAL = "#DD8452"

# Same signature as player_club_vs_national.ipynb (cell 37): default min_min=270
def top_performers_bar(df, metric_col, label, n=15, min_min=270, context_name="Club"):
    sub = df[(df["context"] == context_name) & (df["total_minutes"] >= min_min)]
    top = sub.nlargest(n, metric_col).sort_values(metric_col)
    color = CLR_CLUB if context_name == "Club" else CLR_NATIONAL

    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(top["player"], top[metric_col], color=color, alpha=0.85)
    ax.bar_label(bars, fmt="{:.3f}", padding=3, fontsize=8)
    ax.set_xlabel(label)
    ax.set_title(
        f"Top {n} — {label} ({context_name}, ≥{min_min} min)",
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

In [0]:
# player_club_vs_national.ipynb cells 7 + 13: player_ctx_minutes, inner merge, per-90 cols,
# MIN_MINUTES=270 for dual-context table `dual`.

import numpy as np
from pyspark.sql import functions as F

EVENT_COLS = [
    "shots",
    "xg",
    "passes",
    "passes_completed",
    "passes_att_third",
    "pressures",
    "carries",
    "carries_att_third",
    "duels",
    "dribbles",
    "interceptions",
    "clearances",
    "ball_receipts",
]

evt = spark.table("player_team_match_event_totals").withColumn(
    "player", F.trim(F.col("player").cast("string"))
)

mins = (
    minutes_df.filter(F.col("minutes_played").isNotNull())
    .withColumn("player_name", F.trim(F.col("player_name").cast("string")))
    .groupBy("player_name", "context")
    .agg(
        F.sum("minutes_played").alias("total_minutes"),
        F.countDistinct("match_id").alias("match_count"),
        F.first("position_name", ignorenulls=True).alias("position_name"),
    )
)

mins2 = mins.withColumnRenamed("player_name", "player")
player_ctx = evt.join(mins2, on=["player", "context"], how="inner")

for col in EVENT_COLS:
    player_ctx = player_ctx.withColumn(
        f"{col}_p90",
        F.when(F.col("total_minutes") > 0, F.col(col) / F.col("total_minutes") * F.lit(90.0)).otherwise(F.lit(0.0)),
    )

pdf = player_ctx.toPandas()
pdf["pass_completion_pct"] = pdf["passes_completed"] / pdf["passes"].replace(0, np.nan)

MIN_MINUTES = 270
elig = pdf[pdf["total_minutes"] >= MIN_MINUTES].copy()
elig_club = set(elig.loc[elig["context"] == "Club", "player"])
elig_nat = set(elig.loc[elig["context"] == "National", "player"])
dual_players = elig_club & elig_nat
dual = elig[elig["player"].isin(dual_players)].copy()

print(f"Players with {MIN_MINUTES}+ min in BOTH contexts : {len(dual_players):,}")
print(f"Total rows (2 per player)                       : {len(dual):,}")
print(dual["context"].value_counts())

In [0]:
top_performers_bar(dual, "xg_p90", "xG / 90", n=15, context_name="National")
top_performers_bar(dual, "pressures_p90", "Pressures / 90", n=15, context_name="National")

top_performers_bar(dual, "xg_p90", "xG / 90", n=15, context_name="Club")
top_performers_bar(dual, "pressures_p90", "Pressures / 90", n=15, context_name="Club")

### Export `player_ctx` to BigQuery (Looker / dashboards)

Export the **`player_ctx`** DataFrame from the cell above (player × Club/National, minutes, raw counts, and `*_p90` columns). The code adds **`pass_completion_pct`** before export. Run that pipeline cell **first**.

**Databricks Serverless:** Spark `write.format("bigquery")` is **not allowed** (`UNSUPPORTED_DATA_SOURCE_WRITE` — only csv, json, delta, parquet, … are supported for writes). Use **Option B** (pandas + `google-cloud-bigquery`) on Serverless, **or** attach a **classic** cluster / job cluster with the BigQuery connector for **Option A**.

**Other paths:** write **Delta** / **Parquet** to Unity Catalog (Option C) and point Looker at that, or Parquet → GCS → `bq load`.

Pick a **destination dataset** you can write to (e.g. `analytics`).

**Option B credentials (`DefaultCredentialsError`):** The Databricks **BigQuery connection** used for `spark.read` / SQL does **not** set [Application Default Credentials](https://cloud.google.com/docs/authentication/application-default-credentials) for `google-cloud-bigquery` in Python. You must supply a **service account JSON** one of these ways: set **`GOOGLE_APPLICATION_CREDENTIALS`** (or **`BQ_SERVICE_ACCOUNT_JSON_PATH`**) to a key file on the driver; **or** put the JSON string in **Databricks secrets** and set **`BQ_SECRET_SCOPE`** / **`BQ_SECRET_KEY`** in the code cell; **or** locally run **`gcloud auth application-default login`**.

**Option B library:** If `from google.cloud import bigquery` fails, the cell installs `google-cloud-bigquery` via pip; you can also add that PyPI package under **Compute → Libraries** so it is always present.

Run **one** of A / B / C below after setting `PROJECT_ID`, `DATASET`, `TABLE`.

In [0]:
from pyspark.sql import functions as F

PROJECT_ID = "football-capstone-mds-496219"  # from docs/databricks_bigquery_setup.md
DATASET = "analytics"  # BQ dataset you can write to (avoid overwriting raw_statsbomb unless intended)
TABLE = "player_ctx"

out = player_ctx.withColumn(
    "pass_completion_pct",
    F.when(F.col("passes") > 0, F.col("passes_completed") / F.col("passes")),
)

# --- Option A — Spark → BigQuery (classic / pro clusters with spark-bigquery JAR only; NOT Serverless) ---
# out.write.format("bigquery").mode("overwrite").option("writeMethod", "direct").option(
#     "table", f"{PROJECT_ID}.{DATASET}.{TABLE}"
# ).save()

# --- Option B — pandas → BigQuery (works on Serverless if driver has GCP auth + network) ---
# Databricks images often lack google-cloud-bigquery; install once per session if import fails.
try:
    from google.cloud import bigquery
except ImportError:
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "google-cloud-bigquery"],
        stdout=subprocess.DEVNULL,
    )
    for _name in ("google.cloud.bigquery", "google.cloud", "google"):
        sys.modules.pop(_name, None)
    from google.cloud import bigquery

import json
import os

from google.auth.exceptions import DefaultCredentialsError
from google.oauth2 import service_account

# Service account JSON for the Python BigQuery client (separate from the Lakehouse BQ SQL connection).
# Use ONE of: env vars, file path below, or Databricks secret (JSON string).
BQ_SERVICE_ACCOUNT_JSON_PATH = '/Volumes/workspace/default/gcp_keys/bq-sa.json'  # e.g. "/dbfs/FileStore/keys/capstone-sa.json"
BQ_SECRET_SCOPE = None  # e.g. "capstone-gcp"
BQ_SECRET_KEY = None  # e.g. "bq_service_account_json"


def _bigquery_client(project_id: str):
    json_path = (
        os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
        or os.environ.get("BQ_SERVICE_ACCOUNT_JSON_PATH")
        or BQ_SERVICE_ACCOUNT_JSON_PATH
    )
    if json_path and os.path.isfile(json_path):
        creds = service_account.Credentials.from_service_account_file(json_path)
        return bigquery.Client(project=project_id, credentials=creds)
    if BQ_SECRET_SCOPE and BQ_SECRET_KEY:
        _dbu = globals().get("dbutils")
        if _dbu is None:
            try:
                from databricks.sdk.runtime import dbutils as _dbu
            except ImportError:
                _dbu = None
        if _dbu is None:
            raise RuntimeError(
                "BQ_SECRET_SCOPE/BQ_SECRET_KEY are set but dbutils is not available (not on Databricks?)."
            )
        info = json.loads(_dbu.secrets.get(scope=BQ_SECRET_SCOPE, key=BQ_SECRET_KEY))
        creds = service_account.Credentials.from_service_account_info(info)
        return bigquery.Client(project=project_id, credentials=creds)
    return bigquery.Client(project=project_id)


export_df = out.toPandas()
try:
    client = _bigquery_client(PROJECT_ID)
except DefaultCredentialsError as exc:
    raise RuntimeError(
        "No GCP credentials for the BigQuery Python client. Set GOOGLE_APPLICATION_CREDENTIALS or "
        "BQ_SERVICE_ACCOUNT_JSON_PATH to a service account JSON file, or set BQ_SECRET_SCOPE and "
        "BQ_SECRET_KEY to a Databricks secret holding that JSON string. The BQ Lakehouse connection "
        "does not provide ADC for this API. See the markdown above this cell."
    ) from exc
table_id = f"{PROJECT_ID}.{DATASET}.{TABLE}"
job = client.load_table_from_dataframe(
    export_df,
    table_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()
print(f"Loaded {job.output_rows} rows to {table_id}")

# --- Option C — Delta in Unity Catalog (Serverless-friendly; use Looker Databricks connector or sync to BQ separately) ---
# out.write.mode("overwrite").format("delta").saveAsTable("main.analytics.player_ctx")  # adjust catalog.schema
